# 01 - Exploration

Notebooks are for exploration only. Anything worth keeping moves into `src/` and runs through `src/main.py`.

This notebook shows the two bridges back to the pipeline:
1. **Same config** - compose the Hydra config exactly as `main.py` sees it.
2. **Same results** - query MLflow for runs and load a logged model.

Kernel: `uv run python -m ipykernel install --user --name <project>` once, or select `.venv` in VS Code / Jupyter.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # config paths are relative to the project root
sys.path.insert(0, str(ROOT / "src"))

## 1. Compose the config (with optional overrides)

In [ ]:
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

with initialize_config_dir(config_dir=str(ROOT / "conf"), version_base="1.3"):
    cfg = compose(config_name="config", overrides=["data=wine"])

print(OmegaConf.to_yaml(cfg))

## 2. Load data through the pipeline stages

In [ ]:
import pandas as pd

import ingest
import pre_processing

raw_path = ingest.run(cfg)
df = pd.read_csv(raw_path)
df.describe().T

In [ ]:
df[cfg.data.target].value_counts()

## 3. Compare MLflow runs (run `uv run src/main.py` at least once first)

In [ ]:
import mlflow

from tracking import setup_mlflow

experiment_id = setup_mlflow(cfg)
runs = mlflow.search_runs([experiment_id], order_by=["metrics.f1_macro DESC"])
wanted = ["tags.mlflow.runName", "params.model.name", "metrics.accuracy", "metrics.f1_macro"]
cols = ["run_id"] + [c for c in wanted if c in runs]
runs[cols].head(10)

In [ ]:
if len(runs):
    best = mlflow.sklearn.load_model(f"runs:/{runs.iloc[0].run_id}/model")
    splits = pre_processing.split(pre_processing.clean(df), cfg)
    print(best.score(splits.X_test, splits.y_test))